# 1.3 — Problemas de padronização

Este notebook lista, quantifica e exemplifica problemas de padronização na base de solicitações 156 (2016–2018). A unidade principal de contagem é o **registro afetado**; quando relevante, também são exibidas células, ocorrências e grupos de valores equivalentes.

As verificações são exploratórias: uma diferença encontrada não é automaticamente um erro semântico. Em particular, a ordem de componentes de endereço e a presença de documentos são identificadas por heurísticas e devem ser validadas com o dicionário de dados.

In [1]:
from pathlib import Path
import re, glob, unicodedata
from collections import Counter
import pandas as pd
from IPython.display import display

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 100)
BASE = Path('../dados/processed/cleaned_156_2017_2018.csv')
df = pd.read_csv(BASE, sep=';', encoding='utf-8', dtype='string')
N = len(df)
print(f'{N:,} registros e {df.shape[1]} colunas'.replace(',', '.'))
display(df.head(3))

142.221 registros e 21 colunas


,SOLICITACAO,TIPO,ORGAO,DATA,HORARIO,ASSUNTO,SUBDIVISAO,DESCRICAO,LOGRADOURO_ASS,BAIRRO_ASS,...,MEIO_RESPOSTA,OBSERVACAO,SEXO,BAIRRO_CIDADAO,REGIONAL_CIDADAO,DATA_NASC,TIPO_CIDADAO,ORGAO_RESP,RESPOSTA_FINAL,MENSAGEM_RESPOSTA
0,6714627,SOLICITAÇÃO,INSTITUTO DAS CIDADES INTELIGENTES,20/12/2016,11:07:08,ABORDAGEM SOCIAL DE RUA - ADULTO,DORMINDO/CAÍDAS NA RUA,ABORDAGEM SOCIAL DE RUA - ADULTO - DORMINDO/CAÍDAS NA RUA,"DESEMBARGADOR MOTTA, 3548",MERCES,...,NENHUM,<NA>,F,<NA>,<NA>,<NA>,CIDADÃO,FUNDAÇÃO DE AÇÃO SOCIAL,"HOUVE DESLOCAMENTO DA EQUIPE ATÉ O LOCAL, HAVIAM DUAS PESSOAS AS QUAIS FORAM ORIENTADAS E OFERTADO ATENDIMENTO, NO E...",<NA>
1,6638498,SOLICITAÇÃO,INSTITUTO DAS CIDADES INTELIGENTES,13/10/2016,18:26:03,ABORDAGEM SOCIAL DE RUA - ADULTO,OPERAÇÃO INVERNO,ABORDAGEM SOCIAL DE RUA - ADULTO - OPERAÇÃO INVERNO,"DESEMBARGADOR MOTTA, 0",AGUA VERDE,...,EMAIL,NÃO SOUBE INFORMAR NUMERAÇÃO PREDIAL.,F,<NA>,<NA>,<NA>,CIDADÃO,FUNDAÇÃO DE AÇÃO SOCIAL,13 - REALIZADA ABORDAGEM A PESSOA DA SOLICITAÇÃO E A MESMA NÃO ACEITOU ATENDIMENTO.,<NA>
2,6676499,SOLICITAÇÃO,INSTITUTO DAS CIDADES INTELIGENTES,16/11/2016,16:21:54,TRÂNSITO,FISCALIZAÇÃO IMEDIATA,TRÂNSITO - FISCALIZAÇÃO IMEDIATA,"DESEMBARGADOR MOTTA, 0",AGUA VERDE,...,NENHUM,NÃO SOUBE INFORMAR NUMERAÇÃO PREDIAL.,M,<NA>,<NA>,<NA>,CIDADÃO,SECRETARIA MUNICIPAL DE TRÂNSITO,Informamos que o atendimento aos pedidos para fiscalização e verificação de situações de trânsito obedece a ordem de...,<NA>


## Funções auxiliares e escopo

As funções abaixo preservam o valor original para os exemplos e criam chaves normalizadas apenas para comparar representações.

In [2]:
text_cols = df.select_dtypes(include=['string', 'object']).columns.tolist()
def s(col): return df[col].fillna('').astype('string')
def strip_accents(x):
    return ''.join(c for c in unicodedata.normalize('NFKD', str(x)) if not unicodedata.combining(c))
def key(x, accents=True):
    x = unicodedata.normalize('NFKC', str(x)).replace('\u00a0', ' ')
    x = re.sub(r'\s+', ' ', x).strip().casefold()
    return strip_accents(x) if accents else x
def report(label, mask, denominator=N):
    mask = pd.Series(mask, index=df.index).fillna(False)
    q = int(mask.sum())
    print(f'{label}: {q:,} registros ({q/denominator:.2%})'.replace(',', '.'))
    return mask

# Células e registros atingidos por uma expressão regular
def regex_report(pattern, label, cols=text_cols, flags=0):
    hits = pd.DataFrame(False, index=df.index, columns=cols)
    for c in cols: hits[c] = s(c).str.contains(pattern, regex=True, flags=flags, na=False)
    rows = hits.any(axis=1)
    cells = int(hits.to_numpy().sum())
    print(f'{label}: {int(rows.sum()):,} registros, {cells:,} células'.replace(',', '.'))
    return hits, rows

## 1. Inconsistência de maiúsculas/minúsculas, acentuação e valores equivalentes

Um grupo com a mesma chave normalizada e mais de uma grafia original evidencia representações diferentes do mesmo valor. A tabela mostra os grupos mais frequentes.

In [3]:
def variant_groups(col, accents=True):
    x = s(col); tmp = pd.DataFrame({'original': x, 'chave': x.map(lambda v: key(v, accents))})
    tmp = tmp[tmp['original'].ne('')]
    g = tmp.groupby('chave')['original'].agg(lambda z: sorted(set(z)))
    g = g[g.map(len) > 1]
    return g.sort_values(key=lambda z: z.map(len), ascending=False)

variant_summary = []
for c in text_cols:
    g = variant_groups(c, accents=True)
    affected = s(c).map(lambda v: key(v) in set(g.index) if v else False).sum()
    if len(g): variant_summary.append([c, len(g), int(affected)])
print('Grupos de valores equivalentes por caixa, acento, espaço e Unicode:')
display(pd.DataFrame(variant_summary, columns=['coluna','grupos','registros_afetados']).sort_values('registros_afetados', ascending=False))

for c in ['TIPO', 'RESPOSTA_FINAL', 'MEIO_RESPOSTA', 'BAIRRO_CIDADAO']:
    if c in df:
        g = variant_groups(c)
        print(f'\nExemplos em {c}:')
        display(g.head(8).rename('representações').to_frame())

Grupos de valores equivalentes por caixa, acento, espaço e Unicode:


,coluna,grupos,registros_afetados
5,RESPOSTA_FINAL,383,36541
4,BAIRRO_CIDADAO,36,17668
3,OBSERVACAO,278,14948
2,LOGRADOURO_ASS,1300,9620
6,MENSAGEM_RESPOSTA,45,9160
1,DESCRICAO,6,1150
0,ASSUNTO,1,5



Exemplos em TIPO:


,representações
chave,



Exemplos em RESPOSTA_FINAL:


,representações
chave,
solicitacao atendida conforme informacao do distrito de manutencao urbana do boqueirao.,"[SOLICITAÇÃO ATENDIDA CONFORME INFORMAÇÃO DO DISTRITO DE MANUTENÇÃO URBANA DO BOQUEIRÃO., SOLICITAÇÃO ATENDIDA CONF..."
solicitacao atendida,"[SOLICITACAO ATENDIDA, SOLICITAÇAO ATENDIDA, SOLICITAÇÃO ATENDIDA, Solicitação Atendida, Solicitação atendida, solic..."
abordagem realizada. pessoa foi encaminhada para atendimento na casa das mulheres,"[Abordagem realizada. Pessoa foi encaminhada para atendimento na casa das mulheres, Abordagem realizada. Pessoa foi..."
"a unidade de vigilancia de zoonoses realiza avaliacao ambiental para controle de roedores em areas de risco para transmissao da leptospirose (""doenca da urina do rato""). estas areas sao sujeitas a enchentes e alagamentos, apresentam casos confirmados da doenca, tem acumulo de lixo, esgoto a ceu aberto e, geralmente, sao areas de ocupacao irregular. para um efetivo controle dos roedores, algumas orientacoes sao importantes: - retirar a comida do cachorro durante a noite nao deixar acumular as fezes no quintal. - colocar o lixo em saco plastico e usar lixeiras sempre com a tampa fechada. - manter esgoto/fossa/caixa de gordura sempre em condicoes de uso. revisar as tubulacoes para ver se nao ha furos ou estao quebradas. - nao acumular lixo organico (restos de comida) e material reciclavel no quintal. - fechar vaos, frestas, buracos que permitam a passagem de roedores. - conservar os ralos sempre com tampa fixa, com tela, preferencialmente. - manter a caixa d agua sempre fechada e limpa. - manter a grama/mato sempre aparados. atencao especial para a planta conhecida como ""coroa de cristo"" que oferece abrigo ideal para ratos. - remover entulhos e materiais de construcao.lembre-se: ao separar o lixo organico (restos de comida) do reciclavel, encaminhe para a coleta nos horarios corretos, evitando deixar alimento na rua, disponivel para os animais. para realizar todas estas atividades, e sempre importante proteger-se usando luvas e botas. a unidade de vigilancia de zoonoses informa que, de acordo com o codigo de saude do estado do parana, lei no 13.331 de 2001,art. 329 o morador da edificacao em cujo interior ou dependencias forem encontrados focos de vetores e animais reservatorios de doencas infecciosas, bem como animais peconhentos, fica obrigado a adotar as medidas de profilaxia (prevencao) visando sua eliminacao.",[A Unidade de Vigilância de Zoonoses realiza AVALIAÇÃO AMBIENTAL para controle de roedores em áreas de risco para tr...
solicitamos o encaminhamento deste protocolo ao crm-af.,"[SOLICITAMOS O ENCAMINHAMENTO DESTE PROTOCOLO AO CRM-AF., Solicitamos o encaminhamento deste protocolo ao CRM-AF., S..."
removido por terceiros.,"[REMOVIDO POR TERCEIROS., REMOVIDO pOR Terceiros., REMOVIDO por terceiros., Removido por terceiros., removido por te..."
solicitacao atendida.,"[SOLICITACÃO ATENDIDA., SOLICITAÇÃO ATENDIDA., Solicitação Atendida., Solicitação atendida., solicitação atendida.]"
01 - abordagem realizada. pessoa foi encaminhada para atendimento na casa das mulheres,"[01 - Abordagem realizada. Pessoa foi encaminhada para atendimento na CASA DAS MULHERES, 01 - Abordagem realizada. ..."



Exemplos em MEIO_RESPOSTA:


,representações
chave,



Exemplos em BAIRRO_CIDADAO:


,representações
chave,
sao gabriel,"[SAO GABRIEL, SÃO GABRIEL, SãO GABRIEL]"
sao sebastiao,"[SAO SEBASTIAO, SÃO SEBASTIAO, SÃO SEBASTIÃO]"
nacoes ii,"[NACOES II, NAÇOES II, NAÇÕES II]"
almirante tamandare,"[ALMIRANTE TAMANDARE, ALMIRANTE TAMANDARÉ]"
centro,"[CENTRO, centro]"
campo pequeno,"[CAMPO PEQUENO, CAMPO PEQUENO]"
alto boqueirao,"[ALTO BOQUEIRAO, ALTO BOQUEIRÃO]"
colonia rio grande,"[COLONIA RIO GRANDE, COLÔNIA RIO GRANDE]"


### Espaços extras, caracteres de controle e formatação textual

São contados espaços no início/fim, múltiplos espaços internos, tabulações/quebras de linha, caracteres invisíveis e o caractere de substituição `�`.

In [4]:
patterns = {
'espaço no início/fim': r'(^\s|\s$)',
'múltiplos espaços': r' {2,}',
'tabulação/quebra de linha': r'[\t\r\n]',
'caracteres de controle/encoding': r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f\ufffd]',
}
format_masks = {}
for label, pat in patterns.items():
    hits, rows = regex_report(pat, label)
    format_masks[label] = (hits, rows)
    examples = [(i, c, df.at[i,c]) for c in text_cols for i in df.index[hits[c]][:2]]
    display(pd.DataFrame(examples, columns=['índice','coluna','exemplo']).head(6))

/tmp/ipykernel_66998/2945174940.py:18: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  for c in cols: hits[c] = s(c).str.contains(pattern, regex=True, flags=flags, na=False)
/tmp/ipykernel_66998/2945174940.py:18: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  for c in cols: hits[c] = s(c).str.contains(pattern, regex=True, flags=flags, na=False)
/tmp/ipykernel_66998/2945174940.py:18: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  for c in cols: hits[c] = s(c).str.contains(pattern, regex=True, flags=flags, na=False)
/tmp/ipykernel_66998/2945174940.py:18: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  for c in cols: hits[c] = s(c).str.contains(

espaço no início/fim: 0 registros. 0 células


,índice,coluna,exemplo


múltiplos espaços: 15.129 registros. 16.811 células


,índice,coluna,exemplo
0,3966,ASSUNTO,"VIGILÂNCIA SANITÁRIA EM HOSPITAIS, CLÍNICAS E CONSULTÓRIOS"
1,4064,ASSUNTO,SINALIZAÇÃO LUMINOSA E/OU SONORA DA LINHA FÉRREA
2,1674,SUBDIVISAO,RISCO PARA LEPTOSPIROSE/ ROEDORES EM BUEIRO
3,1763,SUBDIVISAO,RISCO PARA LEPTOSPIROSE/ ROEDORES EM BUEIRO
4,1674,DESCRICAO,FAUNA SINANTRÓPICA - RISCO PARA LEPTOSPIROSE/ ROEDORES EM BUEIRO
5,1763,DESCRICAO,FAUNA SINANTRÓPICA - RISCO PARA LEPTOSPIROSE/ ROEDORES EM BUEIRO


tabulação/quebra de linha: 927 registros. 927 células


,índice,coluna,exemplo
0,8544,OBSERVACAO,NÃO INFORMOU COMPLEMENTO DE BAIRRO. \t TRANSVERSAL 1: AMADOR BUENO.
1,14641,OBSERVACAO,NÃO SOUBE INFORMAR DADOS DOS VEÍCULOS.\tRESSALTA QUE EXISTEM VEÍCULOS EM FRENTE AO 340.
2,805,RESPOSTA_FINAL,Informamos que o atendimento aos pedidos para fiscalização e verificação de situações de trânsito obedece a ordem de...
3,834,RESPOSTA_FINAL,Informamos que o atendimento aos pedidos para fiscalização e verificação de situações de trânsito obedece a ordem de...


caracteres de controle/encoding: 0 registros. 0 células


,índice,coluna,exemplo


## 2. Datas e horários

A verificação procura formatos misturados e valores inválidos. Como a base processada está em texto, a leitura sem um formato declarado pode interpretar o mesmo valor de maneiras diferentes.

In [ ]:
date_cols = [c for c in ['DATA', 'DATA_NASC'] if c in df]
date_patterns = {
'dd/mm/aaaa': r'^\d{2}/\d{2}/\d{4}$',
'aaaa-mm-dd': r'^\d{4}-\d{2}-\d{2}$',
'dd-mm-aaaa': r'^\d{2}-\d{2}-\d{4}$',
'outro formato não vazio': r'.+'
}
date_rows = []
for c in date_cols:
    x = s(c); nonempty = x.ne('')
    for fmt, pat in list(date_patterns.items())[:3]: date_rows.append([c, fmt, int(x.str.match(pat, na=False).sum())])
    parsed = pd.to_datetime(x.where(nonempty), format='%d/%m/%Y', errors='coerce')
    print(f'{c}: vazios/nulos={int((~nonempty).sum()):,}; datas inválidas no formato dd/mm/aaaa={int((nonempty & parsed.isna()).sum()):,}'.replace(',', '.'))
display(pd.DataFrame(date_rows, columns=['coluna','formato','células']))

# Horário: padrões de HH:MM:SS, HH:MM e outros
h = s('HORARIO')
for label, pat in [('HH:MM:SS', r'^\d{2}:\d{2}:\d{2}$'), ('HH:FloMM', r'^\d{1,2}:\d{2}$'), ('outro', r'.+')]:
    print('HORARIO', label, int(h.str.match(pat, na=False).sum()))

DATA: vazios/nulos=0; datas inválidas no formato dd/mm/aaaa=0
DATA_NASC: vazios/nulos=2.450; datas inválidas no formato dd/mm/aaaa=0


,coluna,formato,células
0,DATA,dd/mm/aaaa,142221
1,DATA,aaaa-mm-dd,0
2,DATA,dd-mm-aaaa,0
3,DATA_NASC,dd/mm/aaaa,139771
4,DATA_NASC,aaaa-mm-dd,0
5,DATA_NASC,dd-mm-aaaa,0


HORARIO HH:MM:SS 142221
HORARIO HH:MM 0
HORARIO outro 142221


## 3. Unidades de medida e documentos

As unidades são buscadas nos campos textuais, incluindo abreviações e formas por extenso. CPF e RG são potenciais dados pessoais/documentos e aparecem aqui somente como evidência de formatação; não são exibidos integralmente.

In [6]:
unit_pat = r'(?i)\b\d+(?:[,.]\d+)?\s*(?:mm|cm|m|metro(?:s)?|km|quil[oô]metro(?:s)?|kg|quilo(?:s)?|l|litro(?:s)?|un(?:idade)?(?:s)?|%|grau(?:s)?)\b'
unit_hits, unit_rows = regex_report(unit_pat, 'unidades de medida', flags=re.I)
if unit_rows.any():
    ex=[]
    for c in text_cols:
        for i in df.index[unit_hits[c]][:5]: ex.append([i,c,re.findall(unit_pat,str(df.at[i,c]),flags=re.I)])
    display(pd.DataFrame(ex, columns=['índice','coluna','unidades_encontradas']).head(15))

doc_patterns = {
'CPF': r'(?<!\d)(?:\d{3}\.\d{3}\.\d{3}-\d{2}|\d{11})(?!\d)',
'RG': r'(?i)\b(?:RG|R\.G\.)\s*[:=]?\s*[0-9.]{7,12}-?[0-9Xx]\b',
'telefone': r'(?<!\d)(?:\(?\d{2}\)?\s*)?9?\d{4}[- ]?\d{4}(?!\d)',
'e-mail': r'(?i)\b[\w.+-]+@[\w.-]+\.[a-z]{2,}\b'
}
doc_summary=[]
for label, pat in doc_patterns.items():
    hits, rows = regex_report(pat, label, cols=['DESCRICAO','OBSERVACAO','MENSAGEM_RESPOSTA','RESPOSTA_FINAL'])
    doc_summary.append([label, int(rows.sum()), int(hits.to_numpy().sum())])
display(pd.DataFrame(doc_summary, columns=['padrão','registros_afetados','células']))

unidades de medida: 672 registros. 892 células


,índice,coluna,unidades_encontradas
0,1767,SUBDIVISAO,[80CM]
1,1768,SUBDIVISAO,[80CM]
2,5076,SUBDIVISAO,[80CM]
3,9387,SUBDIVISAO,[80CM]
4,9651,SUBDIVISAO,[80CM]
5,1767,DESCRICAO,[80CM]
6,1768,DESCRICAO,[80CM]
7,5076,DESCRICAO,[80CM]
8,9387,DESCRICAO,[80CM]
9,9651,DESCRICAO,[80CM]


CPF: 40 registros. 40 células
RG: 8 registros. 8 células
telefone: 4.643 registros. 4.658 células
e-mail: 162 registros. 162 células


,padrão,registros_afetados,células
0,CPF,40,40
1,RG,8,8
2,telefone,4643,4658
3,e-mail,162,162


## 4. Representações distintas de nulos

Nulos reais são diferenciados de marcadores textuais como `NULL`, `N/A`, `-`, `não informado` e equivalentes. A lista pode ser ajustada conforme o dicionário da fonte.

In [7]:
null_tokens = r'(?i)^(?:null|none|nan|na|n/a|nulo|não informado|nao informado|sem informação|sem informacao|s/?d|s\.?/\s*d\.?|-|--|não se aplica|nao se aplica)$'
null_summary=[]
for c in df.columns:
    x=s(c); real=int(df[c].isna().sum()); token=int(x.str.fullmatch(null_tokens, na=False).sum())
    if real or token: null_summary.append([c,real,token,real+token])
display(pd.DataFrame(null_summary, columns=['coluna','nulos_reais','marcadores_textuais','total_potencialmente_nulo']).sort_values('total_potencialmente_nulo',ascending=False))
print('Marcadores textuais encontrados:', df[text_cols].apply(lambda x: x.str.fullmatch(null_tokens,na=False)).to_numpy().sum())

,coluna,nulos_reais,marcadores_textuais,total_potencialmente_nulo
12,MENSAGEM_RESPOSTA,121314,0,121314
4,OBSERVACAO,68463,0,68463
7,REGIONAL_CIDADAO,5341,0,5341
8,DATA_NASC,2450,0,2450
6,BAIRRO_CIDADAO,2388,0,2388
2,REGIONAL_ASS,42,0,42
11,RESPOSTA_FINAL,26,0,26
1,BAIRRO_ASS,8,0,8
3,MEIO_RESPOSTA,3,0,3
10,ORGAO_RESP,3,0,3


Marcadores textuais encontrados: 0


## 5. Encoding e ordem de campos compostos

A base processada é UTF-8. Também verificamos os arquivos mensais originais em Latin-1 para detectar falha de decodificação, mojibake (`Ã`, `Â`) e `�`. A ordem do endereço é uma heurística: identifica padrões `logradouro, número` e `número, logradouro`, sem afirmar que todo caso seja erro.

In [8]:
raw_files = sorted(Path('../dados/selected').glob('*.csv'))
encoding_rows=[]
for p in raw_files:
    b=p.read_bytes(); utf8_ok=True
    try: b.decode('utf-8')
    except UnicodeDecodeError: utf8_ok=False
    txt=b.decode('latin1',errors='replace')
    encoding_rows.append([p.name, len(b), utf8_ok, txt.count('Ã'), txt.count('Â'), txt.count('�')])
enc = pd.DataFrame(encoding_rows, columns=['arquivo','bytes','decodifica_utf8','mojibake_Ã','mojibake_Â','substituição_�'])
display(enc)
print('Arquivos mensais:',len(enc),'| falham ao decodificar como UTF-8:',int((~enc.decodifica_utf8).sum()))

addr = s('LOGRADOURO_ASS')
logradouro_numero = addr.str.contains(r'(?i)^[^,]+,\s*\d+', regex=True, na=False)
numero_logradouro = addr.str.contains(r'(?i)^\d+\s*,\s*[^,]+', regex=True, na=False)
report('endereço no formato logradouro, número', logradouro_numero)
report('endereço no formato número, logradouro', numero_logradouro)
print('Exemplos da ordem logradouro, número:'); display(df.loc[logradouro_numero,['SOLICITACAO','LOGRADOURO_ASS']].head(5))
print('Exemplos da ordem número, logradouro:'); display(df.loc[numero_logradouro,['SOLICITACAO','LOGRADOURO_ASS']].head(5))

,arquivo,bytes,decodifica_utf8,mojibake_Ã,mojibake_Â,substituição_�
0,2017-01-01_156_-_Base_de_Dados.csv,53501924,False,341220,40619,0
1,2017-02-01_156_-_Base_de_Dados.csv,47662324,False,319788,33750,0
2,2017-03-01_156_-_Base_de_Dados.csv,42529205,False,313888,33010,0
3,2017-04-01_156_-_Base_de_Dados.csv,8192,False,55,3,0
4,2017-06-01_156_-_Base_de_Dados.csv,62859516,False,460807,55538,0
5,2017-07-01_156_-_Base_de_Dados.csv,61390766,False,450164,57564,0
6,2017-08-01_156_-_Base_de_Dados.csv,60863960,False,452132,57909,0
7,2017-09-01_156_-_Base_de_Dados.csv,57506730,False,427972,55272,0
8,2017-10-01_156_-_Base_de_Dados.csv,54700422,False,407490,51774,0
9,2017-11-01_156_-_Base_de_Dados.csv,55144283,False,409048,52356,0


Arquivos mensais: 24 | falham ao decodificar como UTF-8: 23
endereço no formato logradouro. número: 142.016 registros (99.86%)
endereço no formato número. logradouro: 0 registros (0.00%)
Exemplos da ordem logradouro, número:


,SOLICITACAO,LOGRADOURO_ASS
0,6714627,"DESEMBARGADOR MOTTA, 3548"
1,6638498,"DESEMBARGADOR MOTTA, 0"
2,6676499,"DESEMBARGADOR MOTTA, 0"
3,6648756,"VISCONDE DE GUARAPUAVA, 2684"
4,6659329,"VISCONDE DE GUARAPUAVA, 0"


Exemplos da ordem número, logradouro:


,SOLICITACAO,LOGRADOURO_ASS


## Síntese dos achados

As saídas acima constituem a listagem solicitada. Para a entrega, devem ser destacados: (i) colunas/registros com variantes equivalentes, (ii) totais de espaços e caracteres problemáticos, (iii) formatos de data/hora, (iv) unidades e documentos, (v) nulos reais versus textuais, (vi) encoding dos arquivos originais e (vii) ordens de endereço.

Uma etapa posterior de tratamento deve definir regras de negócio antes de substituir os valores: por exemplo, escolher uma grafia canônica, converter datas para `datetime`, normalizar espaços/Unicode e mascarar documentos nos relatórios.